In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.model_selection import train_test_split

In [29]:
df = pd.read_csv('creditcard.csv')

In [30]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [31]:
X = df.drop('Class',axis=1)
y = df.Class

In [32]:
xtrain,xtest,ytrain,ytest = train_test_split(X,y,train_size=0.8,random_state=42)

In [33]:
pipeline = Pipeline(
    steps=[
        ('scaling',StandardScaler()), #Xtrain - > xtrainscaled
        # ('pca_',PCA(n_components=0.95)),#xtrainscaled - >xtrainscaled-reduceddims
        ('kmeans',KMeans(n_clusters=4)), #xtrainscaled-reduceddims->4cols(updatedXtrain)
        ('model',KNeighborsClassifier()) #updatedXtrain,ytrain
    ]
)
pipeline.fit(xtrain,ytrain)


,steps,"[('scaling', ...), ('kmeans', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_clusters,4
,init,'k-means++'
,n_init,'auto'
,max_iter,300


In [34]:
from sklearn.linear_model import LogisticRegression

In [35]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [36]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

u_sampling =RandomUnderSampler(sampling_strategy=1.0)
o_sampling = SMOTE(sampling_strategy=0.1)

X_resample,y_resample = u_sampling.fit_resample(X,y)

y_resample.value_counts()

Class
0    492
1    492
Name: count, dtype: int64

In [38]:
X_resample.duplicated().sum()

np.int64(19)

In [ ]:
X_resample,y_resample = o_sampling.fit_resample(X,y)
y_resample.value_counts()

Class
0    284315
1     28431
Name: count, dtype: int64

In [40]:
pipeline_1 = Pipeline(
    steps=[
        ('scaling',StandardScaler()), #Xscaledvalues
        # ('pca_',PCA(n_components=0.95)),
        ('kmeans',KMeans(n_clusters=2)), #X
        ('model',LogisticRegression())
    ]
)

pipeline_1.fit(xtrain,ytrain)

,steps,"[('scaling', ...), ('kmeans', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_clusters,2
,init,'k-means++'
,n_init,'auto'
,max_iter,300


In [41]:
pred = pipeline_1.predict(xtrain)

In [42]:
from sklearn.metrics import classification_report
print(classification_report(ytrain,pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    227451
           1       0.15      0.03      0.05       394

    accuracy                           1.00    227845
   macro avg       0.57      0.52      0.52    227845
weighted avg       1.00      1.00      1.00    227845



In [43]:
from sklearn.ensemble import IsolationForest

In [44]:
iso_model = IsolationForest(contamination="auto") #'auto'
res = iso_model.fit_predict(X)
set(res)

{np.int64(-1), np.int64(1)}

In [45]:
y

0         0
1         0
2         0
3         0
4         0
         ..
284802    0
284803    0
284804    0
284805    0
284806    0
Name: Class, Length: 284807, dtype: int64